# Multiclass classification using FeedForward Neural Networks
Implementation with Pytorch, testing on the MNIST dataset.


Pyhton 3.12.0

miriamzara@MacBook-Pro-di-Miriam MCP % python3.12 -m pip install torch 

In [43]:
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import json

In [44]:
# --- Data ---
transform = transforms.Compose([transforms.ToTensor(), lambda x: x.view(-1)])  # normalizes values from [0, 255] to [0, 1]
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform) 
test_set  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=1000)

## NN training

In [45]:
# --- Model Parameters ---

N_INPUT = 28*28
N_HIDDEN = 16
N_OUTPUT = 10
namestring = f"mlp_H{N_HIDDEN}_L1"
output_folder = "./pytorch_nets"
#output_folder = "./serial_code_validation"
os.makedirs(output_folder, exist_ok = True)

In [46]:
# --- Model ---
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(N_INPUT, N_HIDDEN)  
        self.fc2 = nn.Linear(N_HIDDEN, N_OUTPUT)

    def forward(self, x, return_intermediate=False):
        x1 = torch.relu(self.fc1(x))  # hidden layer output
        x2 = self.fc2(x1)             # final output
        if return_intermediate:
            return x2, x1
        return x2

model = Net()                               # instantiation of the network

# --- Training setup ---
criterion = nn.CrossEntropyLoss()           # definition of the loss function
optimizer = optim.Adam(model.parameters())  # definition of the otimizer
                                            # takes as argument all the parameters of the model
                                            # which are defined in the __init__() method of class Net()

# --- Train ---
for epoch in range(3):
    for x, y in train_loader:
        optimizer.zero_grad()               # initializes gradients to zero
        loss = criterion(model(x), y)       # compute loss, comparing the current output of
                                            # the model, model(x), and the real label, y.
        loss.backward()                     # loss is not just the vector of losses. It is
                                            # a tensor that tracks the entire computation graph
                                            # that produced it. 
                                            # so you can call directly .backward() on it.
                                            # this method crosses the graph from bottom to top
                                            # computing the gradients. These gradients
                                            # are stored as attributes in the parameter objects

        optimizer.step()                    # reads the gradients and performs the update
    print(f"Epoch {epoch+1} complete")

# --- Test ---
correct = 0
total = 0
model.eval()                                # switch to evaluation mode. for instance, 
                                            # deactivate dropout - if used in the first place. 
                                            # In this very simple network, it is useless. 

with torch.no_grad():                       # avoid the automatic tracking of gradients
                                            # this saves a lot of memory and time, since you do not need it.
                                            # without no_grad(), the call model(x) would automatically
                                            # produce the computational graph.
    for x, y in test_loader:
        logits = model(x)                   # outputs of the output layer
        pred = logits.argmax(1)             # takes the class with the maximum logit 
        correct += (pred == y).sum().item()
        total += y.size(0)

print("Accuracy:", correct / total)

Epoch 1 complete
Epoch 2 complete
Epoch 3 complete
Accuracy: 0.9338


Comments on the code:

1. torchvision.transforms contains routines for common image transformations, including cropping, padding, rotation, color to grayscale, brightness boosting and so on. Transformations can be chained together using transforms.Compose(). Custom transformations can also applied.
2. transforms.ToTensor() Converts a PIL Image or numpy.ndarray (H x W x C) in the range [0, 255] to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0]. PIL (Pillow, previously Python Imaging Library) is a specific format for images. 
3. x.view() reshapes a tensor. Example: 

```{python}
x = torch.randn(3, 4, 2) # total = 24 elements
x.view(12, 2) # new shape: (12,2)
x.view(4, -1) # infers the second dimension automatically, new shape: (4, 6)
x.view(-1) # infers the first dimension automatically, new shape: (24,)
```
In this usage, x.view() takes as input the image, which is a 3d tensor (height, width, n channels) and flattens it to a 1D array.

4. DataLoader() provides an iterable over the dataset. The batch size parameter specifies how many samples should be loaded at once. Pytorch methods support batch processing. This mean that in the evaluating phase you can feed the whole test dataset at once to the network - if not too big - or in batches, in general. For the training phase, the batch size is used for stochastic gradient descent computation. Smaller batch -> noisier gradient descent steps, less computationally expensive. Larger batch -> less noise, more expensive, can get stuck in local minima. 

## Weight export

1. raw data, .bin
2. metadata, .json

Layout for .bin:

```{python}

[num_layers] (int)
[sizes[0], sizes[1], ..., sizes[num_layers-1]] (ints)
Layer 0 weights (float array, row-major)
Layer 0 biases (float array)
Layer 1 weights ...
Layer 1 biases ...
...

```

Layout for .json:

```{json}

{
  "network": "MLP",
  "dtype": "float32",
  "input_dim": 784,
  "output_dim": 10,
  "layers": [
    { "type": "linear", "in": 784, "out": 256, "activation": "relu" },
    { "type": "linear", "in": 256, "out": 10,  "activation": "none" }
  ],
  "layout": "row-major",
  "bias": true,
  "weight_order": "W[out][in]"
}

```

In [47]:
unsigned_int_type = np.uint64
sizes = [N_INPUT, N_HIDDEN, N_OUTPUT]
state_dict = model.state_dict()


#for name, param in state_dict.items():
#    print(name, param.shape)

w1 = state_dict['fc1.weight'].cpu().numpy()   # weights of first fully connected layer
b1 = state_dict['fc1.bias'].cpu().numpy()     # biases of first fully connected layer
w2 = state_dict['fc2.weight'].cpu().numpy()   # second layer weights
b2 = state_dict['fc2.bias'].cpu().numpy()     # second layer biases

#print("w1:\n\n", w1)
#print("b1:\n\n", b1)
#print("w2:\n\n", w2)
#print("b2:\n\n", b2)

# --- Raw data ---

namestring_bin = namestring  + ".bin"
output_path_bin = os.path.join(output_folder, namestring_bin)

with open(output_path_bin, "wb") as f:
    # wb= Writing in Binary mode
    np.array([len(sizes)], dtype=unsigned_int_type).tofile(f)   # num_layers
    np.array(sizes, dtype=unsigned_int_type).tofile(f)          # layer sizes
    w1.flatten().tofile(f)
    b1.flatten().tofile(f)
    w2.flatten().tofile(f)
    b2.flatten().tofile(f)

# --- Metadata ---

namestring_json = namestring  + ".json"
output_path_json = os.path.join(output_folder, namestring_json)

dict_template = {
  "network": "MLP",
  "dtype": "float32",
  "input_dim": N_INPUT,
  "output_dim": N_OUTPUT,
  "layers": [
    { "type": "linear", "in": N_INPUT, "out": N_HIDDEN, "activation": "relu" },
    { "type": "linear", "in": N_HIDDEN, "out": N_OUTPUT,  "activation": "none" }
  ],
  "bias": True,
  "weight_order": "W[out][in]"
}


with open(output_path_json, "w") as f:
    json.dump(dict_template, f, indent=2)

# ___________________________________________

#### Forward pass (for c++ code validation)
To later compare with the manually implemented c++ version

In [48]:
"""
import matplotlib.pyplot as plt
filename = "pytorch_FP.txt"
filepath = os.path.join("./serial_code_validation", filename)


x, y = test_set[0]
output, hidden = model(x, return_intermediate=True)
plt.imshow(x.detach().cpu().reshape(28, 28), cmap="gray")
plt.savefig(os.path.join("./serial_code_validation","pytorch_input.png"))
print("Label =", y)




with open(filepath, "w") as f:
    f.write("# Input\n")
    np.savetxt(f, x.detach().cpu().numpy().flatten()[None, :], fmt='%g')
    f.write("# Hidden Layer output:\n")
    np.savetxt(f, hidden.detach().cpu().numpy().flatten()[None, :], fmt='%g')
    f.write("# Output:\n")
    # format %g chooses the more compact representation between fixed-point (%f) and scientific notation (%e)
    np.savetxt(f, output.detach().cpu().numpy().flatten()[None, :], fmt='%g')

f.close()
"""


'\nimport matplotlib.pyplot as plt\nfilename = "pytorch_FP.txt"\nfilepath = os.path.join("./serial_code_validation", filename)\n\n\nx, y = test_set[0]\noutput, hidden = model(x, return_intermediate=True)\nplt.imshow(x.detach().cpu().reshape(28, 28), cmap="gray")\nplt.savefig(os.path.join("./serial_code_validation","pytorch_input.png"))\nprint("Label =", y)\n\n\n\n\nwith open(filepath, "w") as f:\n    f.write("# Input\n")\n    np.savetxt(f, x.detach().cpu().numpy().flatten()[None, :], fmt=\'%g\')\n    f.write("# Hidden Layer output:\n")\n    np.savetxt(f, hidden.detach().cpu().numpy().flatten()[None, :], fmt=\'%g\')\n    f.write("# Output:\n")\n    # format %g chooses the more compact representation between fixed-point (%f) and scientific notation (%e)\n    np.savetxt(f, output.detach().cpu().numpy().flatten()[None, :], fmt=\'%g\')\n\nf.close()\n'